# 06 — Evaluation

Compare the pipeline's Pleiades resolutions against the existing manual
`pleiades_uris` annotations (~595 articles), used here purely as a gold
standard — they were never fed into the NER or disambiguation steps.

Metrics are computed at the URI level, per article, then aggregated as
micro- and macro-averaged precision/recall/F1. A qualitative review of
false positives, false negatives and perfect matches follows.

## CONFIG

In [ ]:
import logging
from collections import Counter
from pathlib import Path

import jsonlines
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aec_geoparser.evaluation")

DISAMBIGUATED  = Path("../data/results/disambiguated.jsonl")
ARTICLES_CACHE = Path("../data/cache/articles_with_abstract.jsonl")
EVAL_OUT       = Path("../data/results/evaluation.csv")


## Load data and restrict to the gold-annotated scope

Only articles whose original `pleiades_uris` is non-empty are evaluated — these are the manually annotated ground-truth records (~595 expected).

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    """Load a JSONL file into a DataFrame."""
    with jsonlines.open(path) as reader:
        return pd.DataFrame(list(reader))


articles = load_jsonl(ARTICLES_CACHE)
disambiguated = load_jsonl(DISAMBIGUATED)

has_gold = articles["pleiades_uris"].apply(lambda u: isinstance(u, list) and len(u) > 0)
gold_articles = articles.loc[has_gold, ["id", "pleiades_uris"]].rename(columns={"id": "article_id"})

print(f"Gold-annotated articles in scope: {len(gold_articles)}")

pleiades_resolved = disambiguated[disambiguated["gazetteer_source"] == "pleiades"]


## Per-article URI-level metrics

For each gold article: `gold_set` is the set of manually annotated URIs, `pipeline_set` is the set of Pleiades URIs the pipeline resolved for that article. `TP = |gold ∩ pipeline|`, `FP = |pipeline - gold|`, `FN = |gold - pipeline|`.

In [ ]:
pipeline_uris_by_article = (
    pleiades_resolved.groupby("article_id")["pleiades_uri"]
    .apply(lambda s: set(s.dropna()))
    .to_dict()
)

rows = []
for _, row in gold_articles.iterrows():
    article_id = row["article_id"]
    gold_set = set(row["pleiades_uris"])
    pipeline_set = pipeline_uris_by_article.get(article_id, set())

    tp = gold_set & pipeline_set
    fp = pipeline_set - gold_set
    fn = gold_set - pipeline_set

    precision = len(tp) / len(pipeline_set) if pipeline_set else 0.0
    recall = len(tp) / len(gold_set) if gold_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    rows.append({
        "article_id": article_id,
        "gold_count": len(gold_set),
        "pipeline_count": len(pipeline_set),
        "tp": len(tp),
        "fp": len(fp),
        "fn": len(fn),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fp_uris": sorted(fp),
        "fn_uris": sorted(fn),
    })

eval_df = pd.DataFrame(rows)
EVAL_OUT.parent.mkdir(parents=True, exist_ok=True)
eval_df.drop(columns=["fp_uris", "fn_uris"]).to_csv(EVAL_OUT, index=False)
print(f"Saved per-article breakdown to {EVAL_OUT}")
eval_df.head()


## Aggregate metrics

In [ ]:
total_tp = eval_df["tp"].sum()
total_fp = eval_df["fp"].sum()
total_fn = eval_df["fn"].sum()

micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
            if (micro_precision + micro_recall) else 0.0)

macro_precision = eval_df["precision"].mean()
macro_recall = eval_df["recall"].mean()
macro_f1 = eval_df["f1"].mean()

print("Micro-averaged (pooled across all articles):")
print(f"  precision: {micro_precision:.3f}")
print(f"  recall:    {micro_recall:.3f}")
print(f"  F1:        {micro_f1:.3f}")
print()
print("Macro-averaged (mean of per-article scores):")
print(f"  precision: {macro_precision:.3f}")
print(f"  recall:    {macro_recall:.3f}")
print(f"  F1:        {macro_f1:.3f}")


## False positives — pipeline found, not in gold

Top 20 by frequency. High-frequency FPs may indicate gaps in the manual annotation rather than pipeline errors — worth a closer look.

In [ ]:
fp_counter = Counter(uri for uris in eval_df["fp_uris"] for uri in uris)

print("Top 20 FP URIs (pipeline found, not in gold annotation):")
for uri, count in fp_counter.most_common(20):
    print(f"  {count:>3}  {uri}")


## False negatives — in gold, pipeline missed

Top 20 by frequency, with a sample context fragment from the pipeline's NER output where available (helps spot whether the toponym was never extracted or was extracted but resolved to a different/no place).

In [ ]:
fn_counter = Counter(uri for uris in eval_df["fn_uris"] for uri in uris)

# Map URI -> a sample context fragment seen anywhere in the pipeline's output
# (covers cases where the toponym was extracted but resolved elsewhere).
context_by_uri = {}
for _, rec in disambiguated.iterrows():
    uri = rec.get("pleiades_uri")
    if uri and uri not in context_by_uri:
        context_by_uri[uri] = rec.get("context_fragment")

print("Top 20 FN URIs (in gold annotation, pipeline missed):")
for uri, count in fn_counter.most_common(20):
    context = context_by_uri.get(uri, "(no context fragment found in pipeline output)")
    print(f"  {count:>3}  {uri}")
    print(f"        context: {context}")


## Perfect matches

In [ ]:
perfect = eval_df[(eval_df["fp"] == 0) & (eval_df["fn"] == 0) & (eval_df["gold_count"] > 0)]
pct = len(perfect) / len(eval_df) * 100 if len(eval_df) else 0.0

print(f"Perfect matches (TP=all, FP=0, FN=0): {len(perfect)} / {len(eval_df)} ({pct:.1f}%)")
